# EduTune AI — Dataset Validation

Inspect dataset quality and the persisted validation report.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

def load_jsonl(path):
    return pd.read_json(path, lines=True)

print("Project root:", PROJECT_ROOT)

curated = load_jsonl(PROJECT_ROOT / "data/processed/curated_dataset.jsonl")
synthetic = load_jsonl(PROJECT_ROOT / "data/synthetic/synthetic_dataset.jsonl")
with open(PROJECT_ROOT / "data/evaluation/dataset_validation_report.json", encoding="utf-8") as f:
    validation_report = json.load(f)
display(validation_report)


## Required Fields

In [ ]:
required = ["instruction", "response", "category"]
for name, df in [("Curated", curated), ("Synthetic", synthetic)]:
    print(name)
    print("Missing columns:", [c for c in required if c not in df.columns])
    for c in required:
        if c in df.columns:
            print(c, "empty:", df[c].fillna("").astype(str).str.strip().eq("").sum())


## Duplicate and Distribution Checks

In [ ]:
for name, df in [("Curated", curated), ("Synthetic", synthetic)]:
    print(name, "duplicate rows:", df.duplicated().sum())
    for c in ["category", "difficulty", "task_type"]:
        if c in df.columns:
            print(c)
            display(df[c].value_counts().rename("count").to_frame())


## Validation Summary

In [ ]:
summary = []
for name, df in [("Curated", curated), ("Synthetic", synthetic)]:
    summary.append({
        "dataset": name,
        "records": len(df),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "required_columns_present": all(c in df.columns for c in required)
    })
display(pd.DataFrame(summary))
